### Links and Resources
| Concept | Description |
|---|---|
| [`pyspark.sql.functions.udf`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.udf.html) | Wrap a Python function as a Spark column expr |
| [UDF programming guide](https://spark.apache.org/docs/latest/sql-ref-functions-udf-scalar.html) | Scalar UDF reference |
| [Arrow-optimized Python UDFs](https://docs.databricks.com/en/udf/python-arrow.html) | New: opt into Arrow batching for plain UDFs |
| [Why UDFs are slow](https://databricks.com/blog/2017/10/30/introducing-vectorized-udfs-for-pyspark.html) | Serialization cost explained |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# Python UDFs

A Python UDF wraps a regular Python function so it can be used as a Spark column expression. It's the **escape hatch** for logic that has no built-in equivalent: domain rules, third-party libraries, custom regex parsers.

The catch is performance. For every row, Spark has to:

1. Serialize the columns from the JVM to a Python worker.
2. Run the Python function.
3. Serialize the result back.

That's ~5–50× slower than a built-in. **Try a built-in first; reach for a UDF only when you must.**

In [ ]:
df = spark.createDataFrame([
    (1, "Alice",   "alice@example.com",  None),
    (2, "BOB",     "bob@example.org",    "1990-05-15"),
    (3, "Charlie", "charlie@x.io",       "1985-12-03"),
    (4, "Dana",    None,                  "1992-08-21"),
    (5, "  erin ", "ERIN@example.com",   "2000-01-01"),
], ["id", "name", "email", "dob"])
display(df)

## Defining a UDF — three styles

All three produce the same thing. Pick whichever reads best.

In [ ]:
from pyspark.sql.types import StringType, IntegerType

# Style 1: F.udf wrapping an existing function
def domain_of(email):
    if email is None:
        return None
    return email.lower().split("@")[-1]

domain_udf = F.udf(domain_of, StringType())

# Style 2: F.udf as a decorator with explicit type
@F.udf(StringType())
def name_clean(name):
    if name is None:
        return None
    return name.strip().title()

# Style 3: Register with spark.udf for SQL use
spark.udf.register("domain_sql", domain_of, StringType())

df.select(
    name_clean("name").alias("clean"),
    domain_udf("email").alias("domain"),
).show()

## NULL handling — your problem, not Spark's

Built-in functions usually propagate NULL automatically. UDFs do not. **Your function will be called with `None` arguments** unless you guard or annotate with `@udf(returnType, useArrow=...)` etc. Forgetting to handle `None` is the #1 UDF bug.

In [ ]:
@F.udf(IntegerType())
def name_length_unsafe(name):
    return len(name)               # raises on None

@F.udf(IntegerType())
def name_length_safe(name):
    return None if name is None else len(name)

# The unsafe one will fail at execution
try:
    df.select(name_length_unsafe("email")).show()
except Exception as e:
    print("crashed:", type(e).__name__)

df.select("email", name_length_safe("email").alias("len")).show()

## Return types — strict, not inferred

Spark needs the return type at registration. If your function returns the wrong type, you'll get a confusing error or NULL. Use the `pyspark.sql.types` classes:

| Python | Spark type |
|---|---|
| `int` | `IntegerType()` / `LongType()` |
| `float` | `DoubleType()` |
| `str` | `StringType()` |
| `bool` | `BooleanType()` |
| `list[T]` | `ArrayType(T)` |
| `dict` | `MapType(K, V)` |
| `dataclass` / tuple | `StructType([...])` |

In [ ]:
from pyspark.sql.types import ArrayType, StructType, StructField

@F.udf(ArrayType(StringType()))
def split_words(s):
    return None if s is None else s.lower().split()

@F.udf(StructType([
    StructField("year",  IntegerType()),
    StructField("month", IntegerType()),
    StructField("day",   IntegerType()),
]))
def parse_dob(s):
    if s is None:
        return None
    y, m, d = s.split("-")
    return (int(y), int(m), int(d))

df.select(
    "name",
    split_words("name").alias("words"),
    parse_dob("dob").alias("parts"),
).show(truncate=False)

## Using a UDF in SQL

Registered UDFs (`spark.udf.register`) are reachable from `spark.sql(...)`. Useful when the rest of the query is more readable in SQL.

In [ ]:
df.createOrReplaceTempView("people")

spark.sql("""
SELECT id, name, email, domain_sql(email) AS domain
FROM people
WHERE email IS NOT NULL
""").show()

## Determinism flag

By default Spark assumes UDFs are *deterministic* and may cache or reorder calls. If your UDF reads a clock, calls a service, or uses randomness, mark it non-deterministic:

In [ ]:
import random

@F.udf(IntegerType())
def coin_flip():
    return random.randint(0, 1)

# Without this flag, Spark may evaluate it once per partition and reuse
coin_flip = coin_flip.asNondeterministic()

df.select("name", coin_flip().alias("flip")).show()

## Arrow-optimized Python UDFs (Databricks Runtime 14.3+)

A newer middle ground: keep the simple `@udf` decorator, but Spark uses Arrow batching under the hood. Often 2–3× faster than classic UDFs with zero code change.

```python
@F.udf(StringType(), useArrow=True)
def domain_of_arrow(email):
    return None if email is None else email.split("@")[-1]
```

Notebook 12 covers the more powerful **Pandas UDFs**, which are vectorized end-to-end.

## When *not* to use a UDF

| You wrote | Use this built-in instead |
|---|---|
| `lambda s: s.upper()` | `F.upper("s")` |
| `lambda s: s.split("@")[-1]` | `F.substring_index("s", "@", -1)` |
| `lambda x: round(x, 2)` | `F.round("x", 2)` |
| `lambda d: d.year` | `F.year("d")` |
| `lambda x: None if x is None else x * 2` | `F.col("x") * 2` (NULL propagates automatically) |

Quick rule: if you can express it with `F.*`, do.

## Takeaways

- UDFs are 5–50× slower than built-ins because of JVM↔Python serialization. Use them only when no built-in fits.
- You **must** register the return type and you **must** handle `None`.
- Use `spark.udf.register` to expose the function to SQL.
- Mark non-deterministic UDFs with `.asNondeterministic()` so Spark doesn't cache calls.
- On modern DBR, the `useArrow=True` flag gives a free speedup.
- For heavy logic, jump straight to **Pandas UDFs** (notebook 12).